# 23. Pick a Reference Timestamp for Population Counting

<span style="font-family: 'Courier New', monospace;">

*AI-generated draft (Claude, Anthropic) — for review. All parameters and figures are derived from version-controlled scripts and data.*

The goal is to pick **one fixed second** within Scene 1 (305–320s) that best represents the vent — clearest view, most worms visible, no camera blur or obstruction. Once chosen, every video across the dataset will have a frame extracted at that exact timestamp, giving a consistent, directly-comparable snapshot for population counting.

**Instructions:**
1. Run the cell below — it extracts frames at every second of Scene 1 from one reference video and displays them in a grid
2. Look at the grid and pick the second that looks clearest and has the most worms visible
3. Note that timestamp — you'll use it in notebook `24_count_timeseries.ipynb`

</span>

In [ ]:
import subprocess
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

# ── Configuration ──────────────────────────────────────────────────
# Reference video to use for timestamp exploration
# Pick any standard-cadence video you know has good worm visibility
REFERENCE_VIDEO = Path(
    "/home/jovyan/ooi/san_data/RS03ASHS-PN03B-06-CAMHDA301/"
    "2024/10/01/CAMHDA301-20241001T031500.mp4"
)

SCENE1_START = 305   # seconds — start of Scene 1
SCENE1_END   = 320   # seconds — end of Scene 1
# ──────────────────────────────────────────────────────────────────

timestamps = list(range(SCENE1_START, SCENE1_END + 1))  # 305–320 inclusive

print(f"Extracting {len(timestamps)} frames from {REFERENCE_VIDEO.name} ...")

frames = {}
with tempfile.TemporaryDirectory() as tmpdir:
    for t in timestamps:
        out = Path(tmpdir) / f"frame_{t:04d}.png"
        result = subprocess.run(
            [
                "ffmpeg", "-y",
                "-ss", str(t),
                "-i", str(REFERENCE_VIDEO),
                "-frames:v", "1",
                "-q:v", "2",
                str(out),
            ],
            capture_output=True,
            timeout=30,
        )
        if out.exists():
            frames[t] = mpimg.imread(str(out))
        else:
            print(f"  WARNING: failed to extract t={t}s")

print(f"Extracted {len(frames)} frames. Displaying grid ...\n")

# ── Display grid ────────────────────────────────────────────────────
n = len(frames)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3.5))
axes = axes.flatten()

for i, (t, img) in enumerate(sorted(frames.items())):
    axes[i].imshow(img)
    axes[i].set_title(f"t = {t}s", fontsize=13, fontweight="bold")
    axes[i].axis("off")

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    f"Scene 1 frames — {REFERENCE_VIDEO.name}\n"
    "Pick the timestamp with the clearest view and most worms visible",
    fontsize=14, y=1.01,
)
plt.tight_layout()
plt.show()

print("\nOnce you've chosen your timestamp, note it down and open 24_count_timeseries.ipynb.")